In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(groq_api_key = "", model_name = "llama-3.3-70b-versatile")
print(llm)

profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True} client=<groq.resources.chat.completions.Completions object at 0x0000023B4F1917F0> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000023B4F192510> model_name='llama-3.3-70b-versatile' model_kwargs={} groq_api_key=SecretStr('**********')


In [2]:
# a tool that does multiply
def multiplyTool(a:int, b:int) -> int:
   """Multiply a and b

   Args:
        a(int) first int
        b(int) second int

    Returns
        int: output int   
   """
   return a * b

In [ ]:
from langchain_tavily import TavilySearch
import os

os.environ["TAVILY_API_KEY"] = ""
toolSearch = TavilySearch(max_results = 2)

#creating an array of such tools and binding with llm
tools = [toolSearch, multiplyTool]
llm_with_tool = llm.bind_tools(tools)

In [4]:
from langgraph.graph import MessagesState
from langchain_core.messages import HumanMessage, SystemMessage

# system message
sys_msg = SystemMessage(content="You are a helpful assistant tasked with using search and perfrom multiplication.")


In [5]:
# node
def chatBot(state:MessagesState):
    return {"messages":[llm_with_tool.invoke([sys_msg] + state["messages"])]}
    
    

In [6]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition

# Graph
graph_builder = StateGraph(MessagesState)


In [7]:
# Add nodes
graph_builder.add_node("chatbot", chatBot)
graph_builder.add_node("tools", ToolNode(tools))

In [8]:
# Add Edges
graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")



graph = graph_builder.compile()

In [9]:
from IPython.display import Image, display
try:
    display(Image(graph.get_graph(xray=true).draw_mermaid_png()))
except Exception:
    pass

In [10]:
while True:
  user_input = input("User: ")
  if user_input.lower() in ['quit', 'q']:
    print("Chat Ended")
    break
  events = graph.stream({"messages": ("user", user_input)}, stream_mode="values")
  for event in events:
    event["messages"][-1].pretty_print()

User:  what is langgraph


================================ Human Message =================================

what is langgraph
================================== Ai Message ==================================
Tool Calls:
  tavily_search (cr6pskb0d)
 Call ID: cr6pskb0d
  Args:
    query: langgraph
    topic: general
================================= Tool Message =================================
Name: tavily_search

{"query": "langgraph", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://github.com/langchain-ai/langgraph", "title": "langchain-ai/langgraph: Build resilient language agents as graphs.", "content": "You signed in with another tab or window. You signed out in another tab or window. docs.langchain.com/oss/python/langgraph/. # langchain-ai/langgraph. ## Repository files navigation. Trusted by companies shaping the future of agents – including Klarna, Replit, Elastic, and more – LangGraph is a low-level orchestration framework for building, managing, and deploying lon

User:  multiply 3 and 4


================================ Human Message =================================

multiply 3 and 4
================================== Ai Message ==================================
Tool Calls:
  multiplyTool (gp6tq82rp)
 Call ID: gp6tq82rp
  Args:
    a: 3
    b: 4
================================= Tool Message =================================
Name: multiplyTool

12
================================== Ai Message ==================================

The result of multiplying 3 and 4 is 12.


User:  what is the double of Brad Pitt's age?


================================ Human Message =================================

what is the double of Brad Pitt's age?
================================== Ai Message ==================================
Tool Calls:
  tavily_search (8zccbp51n)
 Call ID: 8zccbp51n
  Args:
    query: Brad Pitt age
    topic: general
================================= Tool Message =================================
Name: tavily_search

{"query": "Brad Pitt age", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.instagram.com/reel/DSdLju1CDv3/", "title": "BRAD PITT TURNS 62 — AND HE'S HITTING PAUSE The Oscar ...", "content": "Hollywood icon Brad Pitt celebrated his 62nd birthday on December 18th. And sources say he's never been better. The Oscar winning actor has had", "score": 0.9993333, "raw_content": null}, {"url": "https://www.facebook.com/thegoodfilms/posts/brad-pitt-enjoying-a-vacation-in-italy-hard-to-believe-hes-61-years-old-/1052032383761602/", "title": "Brad 

BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=multiplyTool={"a": 62, "b": 2}</function>'}}